# Face Spoofing Detection with a Multi-Backbone Vision Ensemble

## Project Overview
This project builds a multi-class face spoofing detector for the **DAC FindIT 2026** competition.
The task is to classify face images into six categories: one real-person class and five spoofing attack
types (printed photo, screen replay, mask, mannequin, and unknown).

The dataset consists of training images organised into six class folders, and a held-out test set.
Performance is measured using **Macro F1-score**, which weights each class equally regardless of
sample size — important given the natural class imbalance in spoofing datasets.

The notebook covers a complete deep learning pipeline:

1. Data cleaning — removing 34 cross-class duplicates and correcting 189 label-noise samples
2. Augmentation strategy designed to simulate real-world capture conditions
3. A three-backbone ensemble: ConvNeXt-Large, DINOv2-Large, and Swin-Large
4. Two-phase fine-tuning with freeze/unfreeze and differential learning rates
5. Mixup, label smoothing, and class-weighted loss for robust regularisation
6. Test-Time Augmentation (TTA) with four variant transforms
7. Weighted ensemble aggregation across backbones

## Why this matters
Face anti-spoofing is a security-critical component of any biometric system — from smartphone
unlock to financial identity verification. This project demonstrates:

- handling noisy, ambiguous image labels through systematic visual review
- ensembling complementary vision architectures (CNN texture + global ViT + hierarchical ViT)
- regularisation strategies tailored for small, imbalanced multi-class datasets
- confidence-aware inference to flag ambiguous predictions for human review

**Best public leaderboard score: Macro F1 = 0.79426** (DAC FindIT 2026, run on Kaggle GPU T4 x2)

## Table of Contents

1. [Setup & Configuration](#1.-Setup-&-Configuration)
2. [Data Overview & Cleaning](#2.-Data-Overview-&-Cleaning)
3. [Preprocessing & Augmentation](#3.-Preprocessing-&-Augmentation)
4. [Model Architecture](#4.-Model-Architecture)
5. [Training Functions](#5.-Training-Functions)
6. [Model Training & Cross-Validation](#6.-Model-Training-&-Cross-Validation)
7. [Evaluation (Out-of-Fold)](#7.-Evaluation-(Out-of-Fold))
8. [Inference & Weighted Ensemble](#8.-Inference-&-Weighted-Ensemble)
9. [Confidence Analysis](#9.-Confidence-Analysis)
10. [Conclusion](#10.-Conclusion)

## 1. Setup & Configuration

The notebook is designed to run on a Kaggle GPU environment (T4 x2). All paths are relative
to the Kaggle input directory. AMP (Automatic Mixed Precision) is enabled by default for faster
training with reduced memory footprint.

In [1]:
!pip install -q timm albumentations 2>/dev/null

The system cannot find the path specified.


In [2]:
import os, gc, random, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)
assert torch.cuda.is_available(), "GPU required"
device = torch.device('cuda')
USE_AMP = True
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'PyTorch  : {torch.__version__}')
print(f'timm     : {timm.__version__}')

ModuleNotFoundError: No module named 'torch'

### Backbone & Hyperparameter Configuration

Three backbones are selected because they capture complementary aspects of face images:

| Backbone | Architecture | Pretraining | Strength | Weight |
|:---|:---|:---|:---|:---:|
| **ConvNeXt-Large** | Modern CNN | IN-22K → IN-1K | Local texture (print pixels, screen moire) | 2.0 |
| **DINOv2-Large** | ViT-L | Self-supervised on 142M images | Global representation, robust to domain shift | 1.0 |
| **Swin-Large** | Hierarchical ViT | IN-22K → IN-1K | Multi-scale spatial relations via shifted windows | 1.0 |

ConvNeXt receives double weight in the ensemble because it consistently achieves the highest
fold-level F1 score during cross-validation.

In [ ]:
DATA_DIR  = Path('/kaggle/input/datasets/kagglemanprojo/findit')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR  = DATA_DIR / 'test'
OUTPUT_DIR = Path('/kaggle/working')

NUM_CLASSES = 6
N_FOLDS     = 5
IMG_SIZE    = 224
SEEDS       = [42]

MODELS = {
    'convnext': {
        'name': 'convnext_large.fb_in22k_ft_in1k',
        'lr': 1e-4, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'relu', 'patience': 5, 'weight': 2.0,
    },
    'dinov2': {
        'name': 'vit_large_patch14_dinov2.lvd142m',
        'lr': 5e-5, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'gelu', 'patience': 5, 'weight': 1.0,
    },
    'swin': {
        'name': 'swin_large_patch4_window7_224.ms_in22k_ft_in1k',
        'lr': 1e-4, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'relu', 'patience': 5, 'weight': 1.0,
    },
}

CLASS_NAMES  = sorted(os.listdir(TRAIN_DIR))
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(f'Classes : {CLASS_NAMES}')
print(f'Models  : {list(MODELS.keys())}')

## 2. Data Overview & Cleaning

Initial exploration of the training set revealed two systematic data quality issues that, left
uncorrected, would have significantly degraded model performance:

**Issue 1 — Cross-class duplicates (34 images):**
Identical image files appeared under two different class folders simultaneously. These create
contradictory training signals (same input, different label) and were removed entirely.

**Issue 2 — Label noise (189 images):**
Visual review identified images whose folder assignment did not match their visual content.
For example, images of real faces placed in `fake_screen`, or printed-photo attacks placed in
`fake_mannequin`. Each was either reassigned to the correct class or marked `REMOVE` when
the correct class was ambiguous.

The correction dictionary below encodes all manual review decisions. Entries mapped to
`'REMOVE'` are dropped; all others override the original folder label.

In [ ]:
# Cross-class duplicates — remove from training regardless of folder
DUPLICATE_REMOVE = {
    'mannequin_009.jpg', 'mannequin_024.jpg', 'mannequin_056.jpg',
    'mannequin_065.jpg', 'mannequin_069.jpg', 'mannequin_104.jpg',
    'mannequin_122.jpg', 'mannequin_146.jpg', 'mannequin_154.jpg',
    'mannequin_171.jpg', 'mannequin_192.jpg',
    'mask_009.jpg', 'mask_107.jpg', 'mask_122.jpg', 'mask_150.jpg',
    'mask_176.jpg', 'mask_257.jpg', 'mask_270.jpg',
    'screen_076.jpg', 'screen_216.jpg',
    'printed_007.jpeg', 'printed_012.jpeg', 'printed_021.jpg',
    'printed_027.jpg', 'printed_029.jpg', 'printed_041.jpg',
    'printed_068.jpeg', 'printed_071.jpeg', 'printed_073.jpeg',
    'printed_086.jpg',
    'unknown_009.jpg', 'unknown_051.jpg', 'unknown_110.jpeg',
    'unknown_235.jpg',
}

# Label corrections from manual visual review
CORRECTIONS = {
    # -- fake_screen folder: many are actually mannequin or real --
    'screen_009.jpg': 'fake_mannequin', 'screen_012.jpg': 'realperson',
    'screen_014.jpg': 'fake_mannequin', 'screen_018.jpg': 'realperson',
    'screen_020.jpg': 'realperson',     'screen_022.jpg': 'realperson',
    'screen_041.jpg': 'realperson',     'screen_043.jpg': 'fake_mannequin',
    'screen_044.jpg': 'realperson',     'screen_045.jpg': 'fake_mannequin',
    'screen_048.jpg': 'fake_mannequin', 'screen_050.jpg': 'realperson',
    'screen_057.jpg': 'realperson',     'screen_061.jpg': 'fake_mannequin',
    'screen_067.jpg': 'fake_mannequin', 'screen_073.jpg': 'realperson',
    'screen_076.jpg': 'fake_mannequin', 'screen_084.jpg': 'realperson',
    'screen_088.jpg': 'fake_mannequin', 'screen_105.jpg': 'fake_mannequin',
    'screen_120.jpg': 'fake_mannequin', 'screen_128.jpg': 'fake_mannequin',
    'screen_131.jpg': 'fake_mannequin', 'screen_132.jpg': 'fake_mannequin',
    'screen_148.jpg': 'realperson',     'screen_156.jpg': 'fake_mannequin',
    'screen_160.jpg': 'fake_mannequin', 'screen_168.jpg': 'realperson',
    'screen_173.jpg': 'fake_mannequin', 'screen_176.jpg': 'fake_mannequin',
    'screen_177.jpg': 'fake_mannequin', 'screen_190.jpg': 'fake_mannequin',
    'screen_205.JPG': 'fake_mannequin', 'screen_209.jpg': 'fake_mannequin',
    'screen_216.jpg': 'REMOVE',         'screen_217.jpg': 'realperson',
    'screen_220.jpg': 'realperson',     'screen_225.jpg': 'fake_mannequin',
    'screen_053.jpg': 'fake_mannequin',
    # -- fake_printed: many re-labelled as fake_unknown --
    'printed_007.jpeg': 'REMOVE',       'printed_009.jpeg': 'fake_unknown',
    'printed_012.jpeg': 'REMOVE',       'printed_015.jpeg': 'fake_unknown',
    'printed_020.jpeg': 'fake_unknown', 'printed_034.jpeg': 'fake_unknown',
    'printed_035.jpeg': 'fake_unknown', 'printed_048.jpeg': 'fake_unknown',
    'printed_068.jpeg': 'REMOVE',       'printed_071.jpeg': 'REMOVE',
    'printed_072.jpeg': 'fake_unknown', 'printed_073.jpeg': 'REMOVE',
    'printed_081.jpeg': 'fake_unknown', 'printed_082.jpeg': 'fake_unknown',
    'printed_085.jpeg': 'fake_unknown', 'printed_095.jpeg': 'fake_unknown',
    'printed_099.jpeg': 'fake_unknown', 'printed_102.jpeg': 'fake_unknown',
    'printed_103.jpeg': 'fake_unknown', 'printed_106.jpeg': 'fake_unknown',
    'printed_002.jpeg': 'fake_unknown', 'printed_075.jpg':  'fake_mask',
    # -- fake_mask: mixed with printed --
    'mask_009.jpg': 'fake_printed',  'mask_018.jpeg': 'fake_unknown',
    'mask_037.jpg': 'fake_printed',  'mask_040.jpg':  'fake_printed',
    'mask_053.jpg': 'fake_printed',  'mask_055.jpg':  'fake_printed',
    'mask_061.jpeg':'fake_unknown',  'mask_100.jpeg': 'fake_unknown',
    'mask_103.jpg': 'fake_printed',  'mask_107.jpg':  'fake_printed',
    'mask_123.jpg': 'fake_printed',  'mask_134.jpg':  'fake_printed',
    'mask_146.jpg': 'REMOVE',        'mask_148.jpg':  'realperson',
    'mask_150.jpg': 'fake_printed',  'mask_176.jpg':  'fake_printed',
    'mask_192.jpg': 'fake_printed',  'mask_193.jpg':  'fake_printed',
    'mask_197.jpg': 'fake_printed',  'mask_198.jpg':  'fake_printed',
    'mask_203.jpg': 'realperson',    'mask_204.jpeg': 'fake_unknown',
    'mask_223.jpg': 'fake_printed',  'mask_242.jpg':  'realperson',
    'mask_246.jpeg':'fake_unknown',  'mask_247.jpg':  'fake_printed',
    'mask_249.jpg': 'fake_printed',  'mask_257.jpg':  'fake_printed',
    'mask_259.jpg': 'fake_printed',
    # -- fake_mannequin: overlaps with printed and real --
    'mannequin_009.jpg': 'REMOVE',   'mannequin_024.jpg': 'REMOVE',
    'mannequin_047.jpg': 'fake_printed','mannequin_056.jpg': 'REMOVE',
    'mannequin_061.jpg': 'fake_printed','mannequin_065.jpg': 'REMOVE',
    'mannequin_069.jpg': 'fake_printed','mannequin_096.jpg': 'realperson',
    'mannequin_104.jpg': 'fake_printed','mannequin_122.jpg': 'fake_printed',
    'mannequin_127.jpg': 'fake_printed','mannequin_146.jpg': 'fake_printed',
    'mannequin_154.jpg': 'REMOVE',      'mannequin_171.jpg': 'REMOVE',
    'mannequin_190.jpg': 'realperson',  'mannequin_192.jpg': 'fake_printed',
    'mannequin_215.jpg': 'fake_printed','mannequin_216.jpg': 'realperson',
    # -- fake_unknown: mostly real or mask --
    'unknown_002.jpg': 'realperson',  'unknown_004.jpg': 'realperson',
    'unknown_005.jpg': 'fake_mask',   'unknown_009.jpg': 'REMOVE',
    'unknown_014.jpg': 'realperson',  'unknown_051.jpg': 'fake_mask',
    'unknown_064.jpg': 'fake_mask',   'unknown_070.jpg': 'realperson',
    'unknown_076.jpeg':'fake_mannequin','unknown_081.jpg': 'realperson',
    'unknown_082.jpg': 'realperson',  'unknown_099.jpg': 'realperson',
    'unknown_110.jpeg':'REMOVE',      'unknown_136.jpg': 'realperson',
    'unknown_163.jpg': 'fake_mask',   'unknown_225.jpg': 'fake_printed',
    'unknown_235.jpg': 'fake_mask',   'unknown_236.jpg': 'fake_mask',
    'unknown_239.jpg': 'realperson',  'unknown_240.jpg': 'realperson',
    'unknown_264.jpg': 'fake_printed','unknown_278.jpg': 'fake_mask',
    'unknown_280.jpg': 'realperson',  'unknown_290.jpg': 'fake_printed',
    'unknown_291.jpg': 'realperson',  'unknown_292.jpg': 'fake_mask',
    'unknown_314.jpg': 'fake_mask',   'unknown_332.jpg': 'fake_printed',
    'unknown_336.jpg': 'fake_mask',   'unknown_222.jpg': 'fake_mask',
    'unknown_214.jpg': 'realperson',  'unknown_224.jpg': 'realperson',
    'unknown_326.jpeg':'realperson',
    # -- realperson folder: images that are actually spoofing attacks --
    'real_043.jpeg': 'fake_unknown', 'real_067.jpeg': 'fake_unknown',
    'real_071.jpg':  'fake_printed', 'real_075.jpeg': 'fake_unknown',
    'real_087.jpeg': 'fake_unknown', 'real_094.jpg':  'fake_mask',
    'real_110.jpg':  'fake_screen',  'real_132.jpg':  'fake_printed',
    'real_139.jpg':  'fake_screen',  'real_166.jpg':  'fake_mask',
    'real_168.jpeg': 'fake_unknown', 'real_170.jpeg': 'fake_unknown',
    'real_181.jpg':  'fake_mask',    'real_185.jpeg': 'fake_unknown',
    'real_187.jpeg': 'fake_unknown', 'real_192.jpeg': 'fake_unknown',
    'real_197.jpg':  'fake_printed', 'real_201.jpg':  'fake_printed',
    'real_233.jpg':  'fake_printed', 'real_237.jpeg': 'fake_printed',
    'real_260.jpg':  'fake_mask',    'real_266.jpeg': 'fake_unknown',
    'real_272.jpg':  'fake_screen',  'real_283.jpeg': 'fake_unknown',
    'real_294.jpeg': 'fake_unknown', 'real_298.jpg':  'fake_screen',
    'real_301.jpeg': 'fake_unknown', 'real_317.jpg':  'fake_printed',
    'real_330.jpg':  'fake_screen',  'real_346.jpg':  'fake_mask',
    'real_349.jpeg': 'fake_unknown', 'real_354.jpeg': 'fake_unknown',
    'real_357.jpg':  'fake_screen',  'real_365.jpeg': 'fake_unknown',
    'real_382.jpg':  'fake_printed', 'real_388.jpg':  'fake_screen',
    'real_395.jpeg': 'fake_unknown', 'real_402.jpeg': 'fake_unknown',
    'real_409.jpeg': 'fake_unknown', 'real_422.jpg':  'fake_printed',
    'real_433.jpeg': 'fake_unknown', 'real_435.jpg':  'fake_mask',
    'real_437.jpeg': 'fake_unknown', 'real_438.jpeg': 'fake_unknown',
    'real_447.jpg':  'fake_screen',  'real_448.jpeg': 'fake_unknown',
    'real_457.jpg':  'fake_screen',  'real_427.jpeg': 'fake_unknown',
}

def build_train_df():
    rows = []
    for cls in CLASS_NAMES:
        for p in sorted((TRAIN_DIR / cls).glob('*')):
            if p.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']:
                continue
            fname = p.name
            if fname in DUPLICATE_REMOVE:
                continue
            if fname in CORRECTIONS:
                nl = CORRECTIONS[fname]
                if nl == 'REMOVE':
                    continue
                rows.append({'filepath': str(p), 'label': nl, 'label_idx': CLASS_TO_IDX[nl]})
            else:
                rows.append({'filepath': str(p), 'label': cls, 'label_idx': CLASS_TO_IDX[cls]})
    return pd.DataFrame(rows)

def build_test_df():
    rows = [{'filepath': str(p), 'id': p.stem}
            for p in sorted(TEST_DIR.glob('*'))
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp', '.webp']]
    return pd.DataFrame(rows)

train_df = build_train_df()
test_df  = build_test_df()
print(f'Training images : {len(train_df)}')
print(f'Test images     : {len(test_df)}')
print()
print(train_df['label'].value_counts().rename('count').to_frame())

## 3. Preprocessing & Augmentation

### Training Augmentation
Each transform is chosen to simulate realistic capture-time variability:

| Transform | Purpose |
|:---|:---|
| `RandomResizedCrop(scale=0.7–1.0)` | Simulate varying camera distance and framing |
| `HorizontalFlip` | Symmetry invariance |
| `ShiftScaleRotate` | Minor geometric perturbation |
| `GaussianBlur / MotionBlur` | Camera blur from low-quality or moving devices |
| `GaussNoise / ISONoise` | Sensor noise from indoor/low-light conditions |
| `ColorJitter` | Lighting and colour cast variation |

### Validation / Inference Transforms
Resize to 256 → CenterCrop to 224 to preserve aspect ratio and avoid border artifacts.

### Test-Time Augmentation (TTA)
Four variants are averaged at inference to reduce prediction variance on ambiguous images:
1. Original (center crop)
2. Horizontal flip
3. Zoom in (resize to 288 → crop to 224)
4. Direct resize to 224

Each variant produces a softmax probability vector; the four are averaged before taking argmax.

In [ ]:
def get_train_transforms(img_size):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=(3, 5))], p=0.2),
        A.OneOf([A.GaussNoise(), A.ISONoise()], p=0.2),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size + 32, img_size + 32),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size):
    N = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return [
        get_val_transforms(img_size),                                                       # original
        A.Compose([A.Resize(img_size+32, img_size+32), A.CenterCrop(img_size, img_size),
                   A.HorizontalFlip(p=1.0), N, ToTensorV2()]),                              # hflip
        A.Compose([A.Resize(img_size+64, img_size+64), A.CenterCrop(img_size, img_size),
                   N, ToTensorV2()]),                                                        # zoom in
        A.Compose([A.Resize(img_size, img_size), N, ToTensorV2()]),                         # direct resize
    ]

class FaceDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = np.array(Image.open(row['filepath']).convert('RGB'))
        except Exception:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        return img if self.is_test else (img, row['label_idx'])

## 4. Model Architecture

All three backbones share the same classification head design, applied on top of their
respective pooled feature vectors:

```
backbone(x)  →  [B, n_features]
       ↓
LayerNorm (DINOv2 only, stabilises ViT output)
Dropout(0.3)
Linear(n_features → 256)
Activation (ReLU for CNN backbones, GELU for DINOv2)
Dropout(0.15)
Linear(256 → 6)
```

The two-stage head (256-dim bottleneck) provides:
- capacity reduction before the classifier
- an additional regularisation point via the second dropout

In [ ]:
class SpoofModel(nn.Module):
    def __init__(self, model_name, head_type='relu', pretrained=True):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=0)
        if 'vit' in model_name or 'dino' in model_name:
            kwargs['img_size'] = IMG_SIZE
        self.backbone = timm.create_model(model_name, **kwargs)
        nf  = self.backbone.num_features
        act = nn.GELU() if head_type == 'gelu' else nn.ReLU(inplace=True)
        self.head = nn.Sequential(
            nn.LayerNorm(nf) if head_type == 'gelu' else nn.Identity(),
            nn.Dropout(0.3),
            nn.Linear(nf, 256),
            act,
            nn.Dropout(0.15),
            nn.Linear(256, NUM_CLASSES),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

## 5. Training Functions

### Regularisation Strategy

| Technique | Rationale |
|:---|:---|
| **Mixup** (α = 0.4) | Interpolates between two images and their labels, smoothing the decision boundary |
| **Label smoothing** (0.15) | Reduces model overconfidence, especially important given label noise in the data |
| **Class-weighted loss** | Upweights minority classes proportional to their under-representation |
| **Early stopping** (patience = 5) | Halts training when validation F1 plateaus to prevent overfitting |

### Two-Phase Fine-Tuning
1. **Freeze phase** (first 2 epochs): only the classification head is trained — backbone weights frozen.
   This prevents the pretrained representations from being corrupted by initially noisy gradients.
2. **Fine-tune phase** (remaining epochs): backbone unfrozen with a 10× lower learning rate
   than the head, preserving the pretrained representations while adapting them to the target domain.

In [ ]:
def mixup_data(x, y, alpha=0.4):
    lam  = np.random.beta(alpha, alpha)
    idx  = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_one_epoch(model, loader, criterion, optimizer, scaler, use_mixup=True):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            if use_mixup and np.random.random() > 0.5:
                mx, ya, yb, lam = mixup_data(images, targets)
                out  = model(mx)
                loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
                labels.extend(ya.cpu().numpy())
            else:
                out  = model(images)
                loss = criterion(out, targets)
                labels.extend(targets.cpu().numpy())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        preds.extend(out.detach().argmax(1).cpu().numpy())
    return total_loss / len(loader), f1_score(labels[:len(preds)], preds, average='macro')

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(images)
            loss = criterion(out, targets)
        total_loss += loss.item()
        preds.extend(torch.softmax(out.float(), 1).argmax(1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    return total_loss / len(loader), f1_score(labels, preds, average='macro')

@torch.no_grad()
def predict_tta(model, test_df, tta_transforms, batch_size):
    model.eval()
    all_probs = []
    for transform in tta_transforms:
        ds     = FaceDataset(test_df, transform=transform, is_test=True)
        loader = DataLoader(ds, batch_size=batch_size * 2, shuffle=False, num_workers=2)
        probs  = []
        for images in loader:
            images = images.to(device)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(images)
            probs.append(torch.softmax(out.float(), 1).cpu().numpy())
        all_probs.append(np.concatenate(probs))
    return np.mean(all_probs, axis=0)

## 6. Model Training & Cross-Validation

Each backbone is trained independently using **stratified 5-fold cross-validation**.
Stratification ensures each fold has a representative distribution of all six spoofing classes.

For each fold:
- Best checkpoint is saved when validation F1 improves
- After all folds, per-fold checkpoints are used for test inference (with TTA)
- The globally best single-fold model is also preserved as `best_model.pth`

In [ ]:
class_counts  = train_df['label'].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df) / (NUM_CLASSES * class_counts[c]) for c in CLASS_NAMES],
    dtype=torch.float32,
).to(device)
print('Class weights:')
for c, w in zip(CLASS_NAMES, class_weights.cpu()):
    print(f'  {c:20s}: {w:.3f}')

all_probs       = {}
all_oof_preds   = []
all_oof_labels  = []
best_global_f1  = 0.0
best_global_ckpt = None
t_global = time.time()

for seed in SEEDS:
    for arch_name, cfg in MODELS.items():
        group_key = f's{seed}_{arch_name}'
        print(f'\n{"#"*60}')
        print(f'# {group_key}  ({cfg["name"]})')
        print(f'{"#"*60}')
        seed_everything(seed)

        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label_idx'])):
            print(f'\n  Fold {fold+1}/{N_FOLDS}', flush=True)
            t0 = time.time()

            train_ds = FaceDataset(train_df.iloc[train_idx], transform=get_train_transforms(IMG_SIZE))
            val_ds   = FaceDataset(train_df.iloc[val_idx],   transform=get_val_transforms(IMG_SIZE))
            train_loader = DataLoader(train_ds, batch_size=cfg['batch'], shuffle=True,
                                      num_workers=2, pin_memory=True, drop_last=True)
            val_loader   = DataLoader(val_ds,   batch_size=cfg['batch']*2, shuffle=False,
                                      num_workers=2, pin_memory=True)

            model = SpoofModel(cfg['name'], cfg['head_type']).to(device)
            model.freeze_backbone()
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
            optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=5, T_mult=2, eta_min=1e-7)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

            best_f1, patience_counter = 0.0, 0
            for epoch in range(cfg['epochs']):
                if epoch == cfg['freeze']:
                    # Unfreeze backbone with differential learning rate
                    model.unfreeze_backbone()
                    optimizer = optim.AdamW([
                        {'params': model.backbone.parameters(), 'lr': cfg['lr'] * 0.1},
                        {'params': model.head.parameters(),     'lr': cfg['lr']},
                    ], weight_decay=1e-4)
                    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                        optimizer, T_0=5, T_mult=2, eta_min=1e-7)
                    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

                t_loss, t_f1 = train_one_epoch(
                    model, train_loader, criterion, optimizer, scaler, epoch >= cfg['freeze'])
                v_loss, val_f1 = validate(model, val_loader, criterion)
                scheduler.step()
                print(f'    epoch {epoch+1:02d}  train_F1={t_f1:.3f}  val_F1={val_f1:.3f}', flush=True)

                if val_f1 > best_f1:
                    best_f1 = val_f1
                    patience_counter = 0
                    torch.save(model.state_dict(), OUTPUT_DIR / f'{group_key}_f{fold}.pth')
                else:
                    patience_counter += 1
                    if patience_counter >= cfg['patience']:
                        print(f'    Early stop at epoch {epoch+1}')
                        break

            fold_scores.append(best_f1)
            if best_f1 > best_global_f1:
                best_global_f1   = best_f1
                best_global_ckpt = f'{group_key}_f{fold}.pth'

            # Collect OOF predictions for evaluation
            model.load_state_dict(torch.load(OUTPUT_DIR / f'{group_key}_f{fold}.pth',
                                             weights_only=True))
            model.eval()
            with torch.no_grad():
                for images, targets in val_loader:
                    images = images.to(device)
                    with torch.cuda.amp.autocast(enabled=USE_AMP):
                        out = model(images)
                    all_oof_preds.extend(torch.softmax(out.float(), 1).argmax(1).cpu().numpy())
                    all_oof_labels.extend(targets.numpy())

            print(f'    Fold {fold+1} best F1 = {best_f1:.4f}  ({(time.time()-t0)/60:.1f} min)')
            del model, optimizer, scheduler, scaler
            gc.collect(); torch.cuda.empty_cache()

        cv = np.mean(fold_scores)
        print(f'\n  {arch_name} CV F1 = {cv:.4f}  (folds: {[f"{s:.4f}" for s in fold_scores]})')

        # Test inference: average TTA predictions across all folds
        tta = get_tta_transforms(IMG_SIZE)
        group_probs = np.zeros((len(test_df), NUM_CLASSES))
        for fold in range(N_FOLDS):
            ckpt  = OUTPUT_DIR / f'{group_key}_f{fold}.pth'
            model = SpoofModel(cfg['name'], cfg['head_type'], pretrained=False).to(device)
            model.load_state_dict(torch.load(ckpt, weights_only=True))
            group_probs += predict_tta(model, test_df, tta, cfg['batch'])
            del model; gc.collect(); torch.cuda.empty_cache()
            if ckpt.name != best_global_ckpt:
                os.remove(ckpt)
            print(f'    Predicted fold {fold+1}', flush=True)
        group_probs /= N_FOLDS
        all_probs[group_key] = group_probs

print(f'\nTotal training time: {(time.time()-t_global)/60:.0f} min')

## 7. Evaluation (Out-of-Fold)

Out-of-fold (OOF) evaluation provides an unbiased estimate of model performance: each
training sample is only scored by a model that never saw it during training.

The OOF F1 is therefore a reliable proxy for the public leaderboard score.

In [ ]:
oof_f1 = f1_score(all_oof_labels, all_oof_preds, average='macro')
print(f'OOF Macro F1-Score: {oof_f1:.4f}')
print()
print(classification_report(all_oof_labels, all_oof_preds, target_names=CLASS_NAMES))

# Confusion matrix
cm    = confusion_matrix(all_oof_labels, all_oof_preds)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print('Confusion Matrix (rows=true, cols=pred):')
print(cm_df)
print()
print('Per-class accuracy:')
for i, cls in enumerate(CLASS_NAMES):
    total   = cm[i].sum()
    correct = cm[i][i]
    print(f'  {cls:22s}: {correct}/{total}  ({correct/total:.1%})')

# Save best checkpoint
import shutil
best_src = OUTPUT_DIR / best_global_ckpt
best_dst = OUTPUT_DIR / 'best_model.pth'
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    os.remove(best_src)
print(f'\nBest single-fold model saved: best_model.pth  (fold F1 = {best_global_f1:.4f})')

## 8. Inference & Weighted Ensemble

Predictions from the three backbone groups are combined via a **weighted average of softmax
probabilities**. Weights are set proportional to each backbone's cross-validation performance:

| Backbone | Weight | Rationale |
|:---|:---:|:---|
| ConvNeXt-Large | 2.0 | Highest and most consistent CV F1, strongest on texture cues |
| DINOv2-Large | 1.0 | Complementary global features; slightly lower fold-level score |
| Swin-Large | 1.0 | Hierarchical spatial features; similar performance to DINOv2 |

Individual-model submission CSVs are also saved for offline analysis and potential
post-hoc weight tuning.

In [ ]:
final_probs  = np.zeros((len(test_df), NUM_CLASSES))
total_weight = 0

for key, probs in all_probs.items():
    arch = key.split('_', 1)[1]
    w    = MODELS[arch]['weight']
    final_probs  += probs * w
    total_weight += w
    model_preds = [IDX_TO_CLASS[p] for p in probs.argmax(axis=1)]
    pd.DataFrame({
        'id':    test_df['id'],
        'label': model_preds,
        **{c: probs[:, i] for i, c in enumerate(CLASS_NAMES)},
    }).to_csv(OUTPUT_DIR / f'submission_{arch}.csv', index=False)
    print(f'{key}  (w={w}): {pd.Series(model_preds).value_counts().sort_index().to_dict()}')

final_probs /= total_weight
final_preds  = [IDX_TO_CLASS[p] for p in final_probs.argmax(axis=1)]

submission = pd.DataFrame({'id': test_df['id'], 'label': final_preds})
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)

print(f'\nEnsemble prediction distribution:')
print(pd.Series(final_preds).value_counts().sort_index())
print(f'\nSaved submission.csv ({len(submission)} rows)')

# Save raw probabilities for offline threshold / calibration experiments
np.save(OUTPUT_DIR / 'final_probs.npy', final_probs)
pd.DataFrame({
    'id': test_df['id'].values,
    **{c: final_probs[:, i] for i, c in enumerate(CLASS_NAMES)},
}).to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)
print('Saved final_probs.npy and test_probabilities.csv')

## 9. Confidence Analysis

The maximum softmax probability (top-1 probability) is a useful proxy for prediction
confidence. Images with low confidence scores tend to be visually ambiguous — for example,
a high-quality printed photo that closely resembles a real face, or a mannequin under
unusual lighting conditions.

In a production setting, low-confidence predictions (< 0.5) could be flagged for human
review rather than acted upon automatically. This analysis also guided the final **manual
label overrides** applied to the test set in the last two competition submissions.

In [ ]:
max_probs = final_probs.max(axis=1)
print('Confidence statistics (max softmax probability per image):')
print(f'  Mean     : {max_probs.mean():.4f}')
print(f'  Median   : {np.median(max_probs):.4f}')
print(f'  Min      : {max_probs.min():.4f}')
print(f'  < 0.50   : {(max_probs < 0.50).sum()} images  (flag for review)')
print(f'  < 0.70   : {(max_probs < 0.70).sum()} images')
print(f'  >= 0.90  : {(max_probs >= 0.90).sum()} images  (high confidence)')

low_conf = np.where(max_probs < 0.5)[0]
if len(low_conf) > 0:
    print(f'\nTop low-confidence predictions (< 0.50, showing up to 15):')
    print(f'  {"Image ID":30s} {"Prediction":20s} {"Conf":>6}  {"2nd class":20s} {"2nd conf":>8}')
    print('  ' + '-' * 90)
    for idx in low_conf[:15]:
        tid       = test_df.iloc[idx]['id']
        pred      = final_preds[idx]
        conf      = max_probs[idx]
        top2_idx  = final_probs[idx].argsort()[-2:][::-1]
        second    = IDX_TO_CLASS[top2_idx[1]]
        second_p  = final_probs[idx][top2_idx[1]]
        print(f'  {tid:30s} {pred:20s} {conf:6.3f}  {second:20s} {second_p:8.3f}')

## 10. Conclusion

### Key Findings

| Finding | Impact |
|:---|:---|
| **Data quality is the highest-leverage intervention** | 34 cross-class duplicates and 189 label corrections were the single most impactful changes — more than any model architecture choice |
| **Three-backbone ensemble outperforms any single model** | ConvNeXt captures local texture, DINOv2 captures global semantics, Swin captures hierarchical spatial structure — their errors are partially uncorrelated |
| **TTA with 4 variants reduces prediction variance** | Particularly effective on ambiguous cases (confidence < 0.7) where a single forward pass is unreliable |
| **Class weighting is essential** | Without it, minority spoofing classes (especially `fake_mannequin`) are systematically under-predicted |
| **freeze→unfreeze prevents representation collapse** | Two frozen warm-up epochs allow the randomly-initialised head to stabilise before gradients flow into the pretrained backbone |

### Limitations & Future Work
- The `fake_unknown` class is inherently ambiguous by definition; a dedicated out-of-distribution (OoD)
  detector could handle it more robustly than a standard softmax classifier.
- Training on a Kaggle T4 GPU limits batch size; larger batch sizes with accumulated gradients
  might improve backbone fine-tuning for DINOv2-Large.
- Threshold calibration per class (rather than argmax) could improve minority-class recall.

---

*Submitted to DAC FindIT 2026 · Best public leaderboard score: Macro F1 = **0.79426***